In [1]:
from transformers import (
    WhisperForConditionalGeneration,
    WhisperTokenizer,
)
import evaluate, random, torch
import numpy as np
from sentence_transformers import SentenceTransformer
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

from functions import MyDataset

torch.manual_seed(2024)
random.seed(2024)
np.random.seed(2024)

No sentence-transformers model found with name uer/sbert-base-chinese-nli. Creating a new one with mean pooling.


In [2]:
language = "zh"
task = "transcribe"

# load model
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small", device_map="auto")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language=language, task=task)
model.eval()

# load WER metric
metric = evaluate.load("wer")

# load data
audio_path = "/scratch/users/nus/e1329380/cs5647/crawled/audio"
test_path = "/scratch/users/nus/e1329380/cs5647/crawled/test.json"
clean_audio = True ###
test_data = MyDataset(audio_path, test_path, clean_audio)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False) # if batch size >1, need a collation function that does padding of input_features and label
print(len(test_data))

Total duration of this dataset is 1.5217511111111115 hours
2000


In [3]:
model_sbert = SentenceTransformer('uer/sbert-base-chinese-nli')

def compute_sentence_embeddings(text_list):
    with torch.no_grad():
        sentence_embeddings = model_sbert.encode(text_list)
        
    return sentence_embeddings
    
def compute_metrics(predictions, references):
    """
    evaluates WER and semantic similarity score
    """
    # Compute Word Error Rate (WER)
    wer = 100 * metric.compute(predictions=predictions, references=references)

    # Compute semantic similarity using sentence embeddings
    pred_embeddings = compute_sentence_embeddings(predictions)
    ref_embeddings = compute_sentence_embeddings(references)
    cosine_sim = cosine_similarity(pred_embeddings, ref_embeddings)
    semantic_similarity = torch.tensor(np.diag(cosine_sim).mean()) # torch.tensor(cosine_sim.mean())

    return {"wer": wer, "semantic_similarity": semantic_similarity.item()}

No sentence-transformers model found with name uer/sbert-base-chinese-nli. Creating a new one with mean pooling.


In [4]:
# takes a while to run without batching
predictions = []
references = []

for batch in tqdm(test_loader):
    
    input_features = batch["input_features"]
    labels = batch["labels"]

    # Convert labels to text for ground truth reference
    reference_text = tokenizer.decode(torch.tensor([l.squeeze() for l in labels]), skip_special_tokens=True)
    references.append(reference_text)

    # Perform inference
    with torch.no_grad():
        outputs = model.generate(torch.tensor(input_features).to("cuda"))
        predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(predicted_text)

  0%|                                                                                        | 0/2000 [00:00<?, ?it/s]/var/tmp/pbs.8676933.pbs101/ipykernel_3405037/1342417032.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  outputs = model.generate(torch.tensor(input_features).to("cuda"))
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecod

In [5]:
predictions[:10]

['那麼的餘幣這樣子',
 ' Bum, yogurt, bum, side.',
 '別動把拼啊 剛才掉出來',
 '我做心意做的啊',
 '也慢点 慢 过来呀',
 '要啊,我能夠改名包機啊',
 " you'll get a little ting ting total.",
 '就會更引發發家常',
 '你媽媽趕我走過來走過來走過來',
 '那給我給你這個第三眼啦']

In [6]:
references[:10]

['你爸还说无近忧必有远虑',
 '一对来听听拉尿不拉屎',
 '现在北京工作比较难找',
 '我上辈子是造了什么孽啊',
 '真是不要脸那个男人是谁',
 '以后我一定把你娘俩',
 '我武你讲得清清楚楚',
 '店主就会跟你拉拉家常',
 '其他人跟我去看看一一',
 '你到底是个什么样的人']

In [7]:
# clean audio
print(compute_metrics(predictions, references))

{'wer': 664.8371531966224, 'semantic_similarity': 0.2397773414850235}


In [14]:
# no clean audio
print(compute_metrics(predictions, references))

{'wer': 658.232810615199, 'semantic_similarity': 0.2524496912956238}


In [9]:
(664.8371531966224+658.232810615199)/2

661.5349819059106